In [15]:
import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader,  random_split

import matplotlib.pyplot as plt

In [16]:
train_dir = '/kaggle/input/datasets/thanhsangtrn/grabage-1/DATASET/TRAIN'
test_dir = '/kaggle/input/datasets/thanhsangtrn/grabage-1/DATASET/TEST'
val_dir = '/kaggle/input/datasets/thanhsangtrn/grabage-1/DATASET/VAL'

In [17]:
import torch
import torch.nn as nn

class CNN_1(nn.Module):
    def __init__(self):
        super(CNN_1, self).__init__()
        self.conv1=nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,padding=0)
        self.conv2=nn.Conv2d(in_channels=32,out_channels=32,kernel_size=3,padding=0)
        self.MaxPool1 = nn.MaxPool2d(kernel_size=2,stride=2)
        self.conv3=nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=0)
        self.conv4=nn.Conv2d(in_channels=64,out_channels=64,kernel_size=3,padding=0)
        self.MaxPool2=nn.MaxPool2d(kernel_size=2,stride=2)
        self.conv5 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=0)
        self.conv6 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=0)
        self.MaxPool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(73728, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.MaxPool1(x)

        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.MaxPool2(x)

        x = self.relu(self.conv5(x))
        x = self.relu(self.conv6(x))
        x = self.MaxPool3(x)

        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x=self.sigmoid(self.fc3(x))

        return x



In [18]:
batch_size_=256
num_epochs = 1


In [19]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
full_dataset = datasets.ImageFolder("/kaggle/input/datasets/diungththanh/grabage-fulll/DATASET/FULL", transform=None)

class DatasetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, idx):
        x, y = self.subset[idx]  
        if self.transform:
            x = self.transform(x)  
        return x, y
    def __len__(self):
        return len(self.subset)


In [ ]:



total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

In [ ]:
train_loader = DataLoader(DatasetWithTransform(train_dataset, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(DatasetWithTransform(val_dataset, val_test_transform), batch_size=32, shuffle=False)
test_loader = DataLoader(DatasetWithTransform(test_dataset, val_test_transform), batch_size=32, shuffle=False)

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1 = CNN_1().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)

In [ ]:
train_losses = []
val_losses = []
train_accurate = []
val_accuarate = []
test_accuracy=[]

for epoch in range(num_epochs):
    model1.train()

    total_loss = 0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model1(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs > 0.5).int()
        correct_train += (preds == labels.int()).sum().item()
        total_train += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct_train / total_train

    train_losses.append(train_loss)
    train_accurate.append(train_accuracy)

    model1.eval()
    val_loss_total = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model1(images)
            loss = criterion(outputs, labels)

            val_loss_total += loss.item()

            preds = (outputs > 0.5).int()
            correct_val += (preds == labels.int()).sum().item()
            total_val += labels.size(0)

    val_loss = val_loss_total / len(val_loader)
    val_accuracy = correct_val / total_val

    val_losses.append(val_loss)
    val_accuarate.append(val_accuracy)
    model1.eval()
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model1(images)
            preds = (outputs > 0.5).int()

            correct_test += (preds.squeeze() == labels.int()).sum().item()
            total_test += labels.size(0)

    test_accuracy.append( correct_test / total_test)

    print(f"Epoch {epoch+1}: "
          f"Train Loss={train_loss:.4f}, Train Acc={train_accuracy:.4f}, Test Acc={correct_test / total_test:.4f} | "
          f"Val Loss={val_loss:.4f}, Val Acc={val_accuracy:.4f}")

In [ ]:
def plot_graphs(x1, x2, string):
    plt.plot(x1, label='train_' + string)
    plt.plot(x2, label='val_' + string)
    plt.xlabel("Epochs")
    plt.ylabel(string)
    plt.title(string + " over epochs")
    plt.legend()
    plt.show()

In [ ]:
 plot_graphs(train_losses, val_losses, "Loss")

plot_graphs(train_accurate,val_accuarate,"Accuracy")
plot_graphs(train_accurate,test_accuracy,"Accuracy")

In [ ]:
from torchmetrics.classification import BinaryConfusionMatrix, BinaryRecall

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
confmat = BinaryConfusionMatrix().to(device)
recall = BinaryRecall().to(device)
model1.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model1(images)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()

        preds = preds.view(-1)
        labels = labels.view(-1)

        confmat.update(preds, labels)
        recall.update(preds, labels)

cm = confmat.compute()
rec = recall.compute()

print("Confusion Matrix:\n", cm)
print("Recall:", rec)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
confmat1 = BinaryConfusionMatrix()
recall1 = BinaryRecall()

model1.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model1(images)

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int()

        preds = preds.view(-1)
        labels = labels.view(-1)

        confmat1.update(preds, labels)
        recall1.update(preds, labels)

cm1 = confmat1.compute()
rec1 = recall1.compute()

print("Confusion Matrix:\n", cm1)
print("Recall:", rec1)